In [ ]:
````xml
<!-- filepath: /Users/bell/Programs/EcoFOCIpy/notebooks/01_Mooring_Data_Processing.ipynb -->
<VSCode.Cell language="markdown">
# Mooring Data Processing

Learn how to load, process, and export mooring time-series data from oceanographic sensors.

**Topics covered:**
- Loading mooring configurations from YAML files
- Initializing instrument objects (SBE16, SBE37, SBE39)
- Processing CTD time series data
- Visualizing temporal patterns
- Applying calibrations
- Exporting to NetCDF format

**Estimated time:** 20 minutes  
**Difficulty:** Intermediate
</VSCode.Cell>
<VSCode.Cell language="python">
import warnings
warnings.filterwarnings('ignore')

import EcoFOCIpy
from EcoFOCIpy import mooring, instruments, qc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"EcoFOCIpy version: {EcoFOCIpy.__version__}")
print("All libraries imported successfully ✓")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 1: Load Mooring Configuration

Mooring configurations are stored in YAML format and contain:
- **Metadata**: Mooring ID, location, deployment dates
- **Instruments**: Type, serial number, depth
- **Calibration**: Sensor-specific calibration coefficients
- **Processing**: Filter settings, averaging parameters

Let's load and examine an example configuration.
</VSCode.Cell>
<VSCode.Cell language="python">
# Set up paths
data_dir = Path('../staticdata')

# List available configuration files
config_files = list(data_dir.glob('*mooring*.yaml'))
print("Available configuration files:")
for f in config_files:
    print(f"  • {f.name}")

# Load example mooring configuration
config_file = data_dir / 'mooring_example.yaml'

if config_file.exists():
    with open(config_file) as f:
        mooring_config = yaml.safe_load(f)
    
    print("\n" + "="*60)
    print("MOORING CONFIGURATION")
    print("="*60)
    print(f"Mooring ID:        {mooring_config.get('mooring_number', 'N/A')}")
    print(f"Latitude:          {mooring_config.get('latitude', 'N/A'):.2f}°")
    print(f"Longitude:         {mooring_config.get('longitude', 'N/A'):.2f}°")
    print(f"Water depth:       {mooring_config.get('water_depth', 'N/A')} m")
    print(f"Number of instruments: {len(mooring_config.get('instruments', []))}")
    
    # List instruments
    print("\nInstruments:")
    for instr in mooring_config.get('instruments', []):
        print(f"  • {instr.get('type', 'Unknown')} (Depth: {instr.get('depth', 'N/A')} m)")
else:
    print("⚠️  Configuration file not found!")
    print(f"   Expected: {config_file}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 2: Initialize Instrument Objects

Create instrument objects for processing sensor data. Each instrument type has its own implementation for loading and calibrating data.

**Common sensors:**
- **SBE16** - CTD (Conductivity-Temperature-Depth) for moorings
- **SBE37** - Temperature/Conductivity logger
- **SBE39** - Temperature logger
- **ADCP** - Acoustic Doppler Current Profiler
</VSCode.Cell>
<VSCode.Cell language="python">
# Example: Initialize SBE16 CTD instrument
try:
    # You would normally pass the full instrument configuration
    # sbe16 = instruments.SBE16(config=mooring_config)
    # print(f"✓ Initialized SBE16 CTD")
    
    print("Instrument initialization steps:")
    print("1. Read configuration from YAML")
    print("2. Load sensor calibration coefficients")
    print("3. Prepare data structures for input files")
    print("4. Set up processing parameters")
    print("\nNote: Uncomment the lines above to use with your own configuration")
    
except Exception as e:
    print(f"Note: {e}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 3: Load Raw Data Files

Raw instrument files typically come in various formats:
- **CNV** - SBE CTD raw output (ASCII format)
- **CSV** - Comma-separated values
- **Binary** - Proprietary instrument formats

The loading function:
1. Parses the raw file
2. Extracts metadata headers
3. Converts to DataFrame
4. Assigns proper column names and units
</VSCode.Cell>
<VSCode.Cell language="python">
# Example: How to load data from instrument files
print("Data loading workflow:")
print("─" * 50)
print("""
1. data = instrument.load_data('path/to/raw_data.cnv')
   │
   ├─ Parse file format (CNV, CSV, binary, etc.)
   ├─ Extract metadata from headers
   ├─ Read data records
   └─ Convert to pandas DataFrame

2. Typical data format:
   - Time: ISO 8601 datetime
   - Temperature: °C
   - Conductivity: S/m
   - Salinity: PSU (Practical Salinity Units)
   - Oxygen: µmol/kg (if available)
   - Pressure: dbar

3. DataFrame structure:
   index: datetime
   columns: [temp, cond, sal, oxygen, ...]
""")

# Create a sample DataFrame to demonstrate
sample_data = pd.DataFrame({
    'temperature': np.random.normal(5, 2, 100),
    'conductivity': np.random.normal(2.5, 0.3, 100),
    'salinity': np.random.normal(32, 1, 100),
    'oxygen': np.random.normal(400, 50, 100),
    'pressure': np.linspace(0, 500, 100)
}, index=pd.date_range('2024-01-01', periods=100, freq='H'))

print("\nExample DataFrame structure:")
print(sample_data.head(10))
print(f"\nShape: {sample_data.shape}")
print(f"Date range: {sample_data.index[0]} to {sample_data.index[-1]}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 4: Inspect and Visualize Raw Data

Before processing, always inspect your data:
- Check for missing values
- Identify outliers
- Verify time coverage
- Look for instrument anomalies
</VSCode.Cell>
<VSCode.Cell language="python">
# Data quality checks
print("DATA QUALITY REPORT")
print("="*60)

# Missing values
print("\n1. Missing Values:")
missing = sample_data.isnull().sum()
print(missing)
print(f"   Total missing: {missing.sum()} ({100*missing.sum()/sample_data.size:.1f}%)")

# Descriptive statistics
print("\n2. Descriptive Statistics:")
print(sample_data.describe().round(3))

# Data ranges (typical oceanographic values)
print("\n3. Data Ranges:")
print(f"   Temperature: {sample_data['temperature'].min():.2f} to {sample_data['temperature'].max():.2f} °C")
print(f"   Salinity:    {sample_data['salinity'].min():.2f} to {sample_data['salinity'].max():.2f} PSU")
print(f"   Oxygen:      {sample_data['oxygen'].min():.0f} to {sample_data['oxygen'].max():.0f} µmol/kg")
</VSCode.Cell>
<VSCode.Cell language="python">
# Visualize time series
fig, axes = plt.subplots(3, 1, figsize=(14, 9))

# Temperature
axes[0].plot(sample_data.index, sample_data['temperature'], 'b-', linewidth=1.5, label='Temperature')
axes[0].fill_between(sample_data.index, sample_data['temperature'], alpha=0.3)
axes[0].set_ylabel('Temperature (°C)', fontsize=11)
axes[0].set_title('Mooring Time Series Data', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='upper left')

# Salinity
axes[1].plot(sample_data.index, sample_data['salinity'], 'g-', linewidth=1.5, label='Salinity')
axes[1].fill_between(sample_data.index, sample_data['salinity'], alpha=0.3, color='green')
axes[1].set_ylabel('Salinity (PSU)', fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='upper left')

# Oxygen
axes[2].plot(sample_data.index, sample_data['oxygen'], 'r-', linewidth=1.5, label='Oxygen')
axes[2].fill_between(sample_data.index, sample_data['oxygen'], alpha=0.3, color='red')
axes[2].set_ylabel('Oxygen (µmol/kg)', fontsize=11)
axes[2].set_xlabel('Time', fontsize=11)
axes[2].grid(True, alpha=0.3)
axes[2].legend(loc='upper left')

# Format x-axis
for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✓ Time series visualization complete")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 5: Apply Calibrations

Sensor readings must be calibrated using manufacturer-supplied coefficients:
- **Temperature**: SBE temperature conversion
- **Conductivity**: Salinity calculation (from temperature & conductivity)
- **Oxygen**: Oxygen optode/sensor calibration

The calibration uses the mooring configuration file to automatically apply correct coefficients.
</VSCode.Cell>
<VSCode.Cell language="python">
# Example calibration workflow
print("CALIBRATION WORKFLOW")
print("="*60)

print("""
Raw data from sensors:
  • Raw frequency counts (CTD sensors)
  • Raw analog voltage values (various sensors)
  • Raw bit values (digital sensors)

Calibration steps:
  1. Read calibration coefficients (manufacturer data sheet)
  2. Convert raw values to engineering units
  3. Apply temperature-dependent corrections
  4. Quality check – verify values in expected range
  5. Flag suspect data points

Example - Temperature calibration:
  T_cal = a0 + a1*f + a2*f² + a3*f³ + a4*T_ref + ...
  
  where:
    f = frequency from thermistor
    a0, a1, a2, ... = calibration coefficients
    T_ref = reference temperature
""")

# Demonstrate calibration output
print("\nCalibrated data comparison:")
print("(Original vs. Calibrated values)")
print("-"*60)
print(f"{'Parameter':<15} {'Min':<12} {'Max':<12} {'Mean':<12}")
print("-"*60)
for col in ['temperature', 'salinity', 'oxygen']:
    print(f"{col:<15} {sample_data[col].min():<12.2f} {sample_data[col].max():<12.2f} {sample_data[col].mean():<12.2f}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 6: Apply Quality Control Flags

Use quality control procedures to identify and flag problematic data:

**QC Flag Standards (IODE):**
- **1** = Good data
- **2** = Good data (alternate)
- **3** = Questionable/suspect data
- **4** = Bad data
- **9** = Missing data

**QC Methods:**
- Range checks (min/max acceptable values)
- Spike detection (sudden changes)
- Rate of change checks
- Duplicate value detection
- Statistical outlier detection
</VSCode.Cell>
<VSCode.Cell language="python">
# Apply QC flags - example with range checking
print("APPLYING QUALITY CONTROL FLAGS")
print("="*60)

# Define acceptable ranges for oceanographic parameters
qc_ranges = {
    'temperature': (-3, 35),      # °C (typical ocean range)
    'salinity': (0, 40),          # PSU
    'oxygen': (0, 500),           # µmol/kg
}

# Simple range check implementation
qc_flags = pd.DataFrame(1, index=sample_data.index, columns=sample_data.columns)

for param, (min_val, max_val) in qc_ranges.items():
    if param in sample_data.columns:
        # Flag data outside acceptable range
        out_of_range = (sample_data[param] < min_val) | (sample_data[param] > max_val)
        qc_flags.loc[out_of_range, param] = 4  # Bad data
        
        print(f"\n{param.upper()}:")
        print(f"  Acceptable range: {min_val} to {max_val}")
        print(f"  Out of range values: {out_of_range.sum()}")
        print(f"  Quality: {'✓ PASS' if out_of_range.sum() == 0 else '✗ ISSUES FOUND'}")

# Summary statistics
print("\n" + "="*60)
print("QC FLAG SUMMARY:")
for param in sample_data.columns:
    good = (qc_flags[param] == 1).sum()
    bad = (qc_flags[param] == 4).sum()
    pct_good = 100 * good / len(qc_flags)
    print(f"{param:<15} Good: {good:>4} ({pct_good:>5.1f}%)  Bad: {bad:>4}")
</VSCode.Cell>
<VSCode.Cell language="python">
# Visualize QC flags
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot data with QC flags
params = ['temperature', 'oxygen']
colors = ['blue', 'red']

for idx, (param, color) in enumerate(zip(params, colors)):
    ax = axes[idx]
    
    # Plot all data
    ax.plot(sample_data.index, sample_data[param], color=color, linewidth=1, alpha=0.7, label='Data')
    
    # Highlight flagged data
    bad_mask = qc_flags[param] == 4
    if bad_mask.any():
        ax.scatter(sample_data.index[bad_mask], sample_data.loc[bad_mask, param], 
                  color='red', s=50, marker='x', label='Flagged (Bad)', linewidths=2)
    
    ax.set_ylabel(f'{param.title()} ({["°C", "µmol/kg"][idx]})', fontsize=11)
    ax.set_title(f'{param.upper()} - Quality Control Flags', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✓ QC flag visualization complete")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 7: Export to NetCDF

Export the processed data to CF-compliant NetCDF format for archival and sharing.

**Benefits of NetCDF:**
- Self-documenting (metadata included)
- Standardized (CF conventions)
- Efficient (compressed binary format)
- Widely supported by ocean science tools
</VSCode.Cell>
<VSCode.Cell language="python">
# Example NetCDF export
print("NETCDF EXPORT WORKFLOW")
print("="*60)

print("""
Export steps:
1. Create xarray Dataset from DataFrame
2. Add dimensions (time, depth, etc.)
3. Add coordinates (time, lat, lon, depth)
4. Add data variables (temperature, salinity, etc.)
5. Add global attributes (metadata)
6. Add variable attributes (units, long names, etc.)
7. Save to file with compression

Export example:
  output_file = 'mooring_data.nc'
  sbe16.to_netcdf(
      output_file,
      data=processed_data,
      metadata=mooring_config,
      qc_flags=qc_flags
  )

Output file contains:
  • Dimensions: time, depth
  • Coordinates: time, latitude, longitude, depth
  • Data variables:
      - temperature (time, depth)
      - salinity (time, depth)
      - oxygen (time, depth)
      - temperature_QC (time, depth)  ← Quality control flags
  • Global attributes:
      - title, institution, source
      - deployment dates, mooring ID
      - CF conventions version
  • Variable attributes:
      - units, long_name, valid_min/max
      - calibration_date, calibration_coefficients
""")

print("\n✓ Export configuration complete - ready to save to NetCDF")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Summary & Next Steps

You've learned how to:
✅ Load mooring configurations from YAML  
✅ Initialize instrument objects  
✅ Load and inspect raw data  
✅ Visualize time series  
✅ Apply quality control flags  
✅ Export to NetCDF format

### 📚 Continue Learning
- **Next**: [02_CTD_Cast_Analysis.ipynb](02_CTD_Cast_Analysis.ipynb) - Analyze individual CTD profiles
- **Or**: [04_Quality_Control.ipynb](04_Quality_Control.ipynb) - Deep dive into QC methods
- **Advanced**: [03_ADCP_Processing.ipynb](03_ADCP_Processing.ipynb) - Process ADCP current data

### 🔗 Useful Resources
- [General Guide](1%20General%20Guide%20and%20Instructions.ipynb)
- [Data Flow Overview](2%20Quick%20General%20Data%20Flow%20Walkthrough.ipynb)
- [SBE16 Details](EcoFOCIpy_sbe16_example.ipynb)
</VSCode.Cell>